# NeuralRetail – FastAPI Scoring API

**Amdox Technologies | AMX-DS-2026-04 | April 2026**

Prepared by: **Ayush Tiwari** | Data Science & Analytics

---

## 🎯 What is this notebook?

This notebook **writes, explains, and tests** the `fastapi_app.py` REST API — the **production scoring layer** of NeuralRetail.

### Why FastAPI alongside the Streamlit dashboard?

| | **Streamlit Dashboard** | **FastAPI Scoring API** |
|---|---|---|
| **Who uses it** | Humans (analysts, managers) | Other systems, apps, CRM tools |
| **How accessed** | Browser (click & view) | HTTP requests (code / automation) |
| **Output** | Charts and tables | JSON data |
| **Use case** | Decision making, exploration | Automated scoring, integrations |
| **Conflict?** | ❌ None | ❌ None — both read same Excel files independently |

### Architecture Position

```
rfm_segments_churn.xlsx  ──┐
inventory_eoq.xlsx        ──┼──► fastapi_app.py  (port 8000)  ← REST clients
demand_forecast.xlsx      ──┤
online_retail_CLEANED.xlsx─┘
                            └──► neuralretail_dashboard.py (port 8501) ← Browser
```

---

## 📋 Endpoints Built in This Notebook

| # | Endpoint | Method | Purpose |
|---|---|---|---|
| 1 | `/health` | GET | API health check + data status |
| 2 | `/summary` | GET | Full KPI snapshot |
| 3 | `/predict/churn` | POST | Churn score for 1 customer |
| 4 | `/predict/demand` | POST | Demand forecast for 1 SKU |
| 5 | `/segment/score` | POST | RFM segment for 1 customer |
| 6 | `/inventory/reorder` | POST | EOQ & reorder advice for 1 SKU |
| 7 | `/predict/churn/high-risk` | GET | Top-N high risk customers (batch) |
| 8 | `/inventory/alerts` | GET | All SKUs needing reorder (batch) |

---

**Files needed** (already created by `neuralretail_models_.ipynb`):
- `rfm_segments_churn.xlsx`
- `inventory_eoq.xlsx`
- `demand_forecast.xlsx`
- `online_retail_CLEANED.xlsx`


## STEP 1 – INSTALL DEPENDENCIES

In [1]:
import subprocess, sys

# fastapi       → the web framework (like Flask but modern + async)
# uvicorn       → the ASGI server that actually runs FastAPI
# pydantic      → automatic request/response validation (built into FastAPI)
# requests      → used in STEP 6 to test each endpoint from this notebook
# openpyxl      → reads .xlsx files
libs = ['fastapi', 'uvicorn[standard]', 'pydantic', 'requests', 'openpyxl', 'pandas']
for lib in libs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '-q'])

print('✅  fastapi          installed  — REST API framework')
print('✅  uvicorn          installed  — ASGI server to run FastAPI')
print('✅  pydantic         installed  — request/response validation')
print('✅  requests         installed  — used to test endpoints in Step 6')
print('✅  openpyxl         installed  — reads .xlsx files')
print('✅  pandas           installed  — data handling')


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


✅  fastapi          installed  — REST API framework
✅  uvicorn          installed  — ASGI server to run FastAPI
✅  pydantic         installed  — request/response validation
✅  requests         installed  — used to test endpoints in Step 6
✅  openpyxl         installed  — reads .xlsx files
✅  pandas           installed  — data handling



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


## STEP 2 – VERIFY DATA FILES EXIST

In [2]:
import os, pandas as pd

print('\n' + '='*60)
print('  STEP 2 – CHECKING DATA FILES')
print('='*60)

# These 4 files are outputs of neuralretail_models_.ipynb
# The API reads from them — no model retraining needed at serve time
files = {
    'rfm_segments_churn.xlsx'   : 'Customer segments + churn scores (from XGBoost + LGBM)',
    'inventory_eoq.xlsx'        : 'EOQ + ABC-XYZ inventory classification',
    'demand_forecast.xlsx'      : 'Prophet demand forecasts per SKU',
    'online_retail_CLEANED.xlsx': 'Cleaned transaction data',
}

all_ok = True
for f, desc in files.items():
    exists = os.path.exists(f)
    icon   = '✅' if exists else '❌  MISSING – run neuralretail_models_.ipynb first'
    print(f'  {icon}  {f}')
    print(f'         → {desc}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('  ✅  All files found – ready to build FastAPI')
else:
    print('  ❌  Fix missing files before continuing')


  STEP 2 – CHECKING DATA FILES
  ✅  rfm_segments_churn.xlsx
         → Customer segments + churn scores (from XGBoost + LGBM)
  ✅  inventory_eoq.xlsx
         → EOQ + ABC-XYZ inventory classification
  ✅  demand_forecast.xlsx
         → Prophet demand forecasts per SKU
  ✅  online_retail_CLEANED.xlsx
         → Cleaned transaction data

  ✅  All files found – ready to build FastAPI


## STEP 3 – PREVIEW DATA COLUMNS

Before writing the API, we inspect the exact column names in each file.
The API endpoints are **built to match these column names exactly**, so this step also confirms alignment.

In [3]:
print('\n' + '='*60)
print('  STEP 3 – DATA PREVIEW (columns the API will serve)')
print('='*60)

rfm = pd.read_excel('rfm_segments_churn.xlsx')
inv = pd.read_excel('inventory_eoq.xlsx')
fc  = pd.read_excel('demand_forecast.xlsx') if os.path.exists('demand_forecast.xlsx') else pd.DataFrame()

print(f'\n  📋 rfm_segments_churn  : {rfm.shape[0]:,} rows × {rfm.shape[1]} cols')
print(f'     Columns : {list(rfm.columns)}')

print(f'\n  📋 inventory_eoq       : {inv.shape[0]:,} rows × {inv.shape[1]} cols')
print(f'     Columns : {list(inv.columns)}')

if not fc.empty:
    print(f'\n  📋 demand_forecast     : {fc.shape[0]:,} rows × {fc.shape[1]} cols')
    print(f'     Columns : {list(fc.columns)}')

# ── Save sample IDs for endpoint tests in Step 6 ──
SAMPLE_CUSTOMER = str(rfm['CustomerID'].iloc[0])
SAMPLE_SKU      = str(inv['StockCode'].iloc[0])

print(f'\n  ── Sample values saved for endpoint testing (Step 6) ──')
print(f'  SAMPLE_CUSTOMER : {SAMPLE_CUSTOMER}')
print(f'  SAMPLE_SKU      : {SAMPLE_SKU}')

# ── Check required columns ──
required_rfm = ['CustomerID', 'ChurnProba', 'ChurnRisk', 'Recency', 'Frequency', 'Monetary']
required_inv = ['StockCode', 'EOQ', 'SafetyStock', 'ReorderPoint', 'AvgDailyQty']

print(f'\n  ── Required column check ──')
for col in required_rfm:
    status = '✅' if col in rfm.columns else '⚠️  MISSING'
    print(f'  rfm → {col:20s} {status}')
for col in required_inv:
    status = '✅' if col in inv.columns else '⚠️  MISSING'
    print(f'  inv → {col:20s} {status}')


  STEP 3 – DATA PREVIEW (columns the API will serve)

  📋 rfm_segments_churn  : 5,072 rows × 48 cols
     Columns : ['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'CLV_Estimate', 'AvgOrderValue', 'ActiveLast7', 'ActiveLast14', 'ActiveLast30', 'ActiveLast60', 'AvgDaysBetweenOrders', 'StdDaysBetweenOrders', 'MaxGapDays', 'MinGapDays', 'RevLast30', 'RevPrev30', 'OrdersLast30', 'OrdersPrev30', 'RevTrend', 'OrderTrend', 'RevTrendPct', 'OrderValueStd', 'OrderValueMax', 'OrderValueMin', 'TotalOrders', 'TotalItems', 'UniqueProducts', 'AvgItemsPerOrder', 'TotalRevenue', 'AvgCompPrice', 'AvgPriceDiff', 'RainyDayOrders', 'RevenuePerOrder', 'ItemsPerOrder', 'ProductDiversity', 'Age', 'AgeGroup', 'Region', 'Gender', 'LoyaltyTier', 'LoyaltyTierNum', 'Churned', 'ChurnProba', 'ChurnRisk']

  📋 inventory_eoq       : 4,249 rows × 19 cols
     Columns : ['StockCode', 'AvgDailyQty', 'StdDailyQty', 'TotalQty', 'TotalRev', 'AvgPrice', 'AvgCompPrice', 'Annua

## STEP 4 – WRITE `fastapi_app.py`

This cell writes the complete FastAPI application to disk.

### How the API works internally

```
Server starts → load_data() reads all 4 Excel files into RAM once
                                                    ↓
POST /predict/churn  →  Pydantic validates JSON  →  lookup customer in rfm df  →  return JSON
POST /predict/demand →  Pydantic validates JSON  →  filter forecast df by SKU  →  return JSON
POST /segment/score  →  Pydantic validates JSON  →  lookup customer in rfm df  →  return JSON
POST /inventory/... →   Pydantic validates JSON  →  lookup SKU in inv df       →  return JSON
```

### Why load data once at startup?

Reading an Excel file takes ~0.5 seconds. If we loaded on every request:
- 100 simultaneous requests = **50 seconds** of file I/O

Loading once at startup → all requests answer **from RAM in < 5ms**.

### What is Pydantic validation?

```python
# If someone sends: { "customer_id": 99999 }  (integer, not string)
# Pydantic automatically returns:
# HTTP 422 Unprocessable Entity
# { "detail": "customer_id: str type expected" }
# Your code never even runs — the bad request is rejected at the gate
```


In [4]:
# This cell writes fastapi_app.py to your project folder
# It contains the complete production-ready FastAPI application

api_lines = []

api_lines.append("""# ================================================================
#   NeuralRetail – FastAPI Scoring API
#   Amdox Technologies | AMX-DS-2026-04 | April 2026
#   Prepared by: Ayush Tiwari | Data Science & Analytics
#
#   HOW TO RUN (in a terminal):
#   uvicorn fastapi_app:app --reload --port 8000
#
#   Interactive API docs auto-open at:
#   http://localhost:8000/docs
# ================================================================

from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field
from typing import Optional, List
import pandas as pd
import numpy as np
import os
from datetime import datetime

# ── App Setup ────────────────────────────────────────────────
app = FastAPI(
    title       = "NeuralRetail API",
    description = "AI-Powered Sales Intelligence — Amdox Technologies",
    version     = "1.0.0",
    contact     = {"name": "Ayush Tiwari"},
)

# ── Load All Data at Startup (once, into RAM) ─────────────────
# This means every request reads from memory, not disk
# Result: <5ms response time vs ~500ms if we read Excel per request
def load_data():
    data  = {}
    files = {
        "rfm"      : "rfm_segments_churn.xlsx",
        "inventory": "inventory_eoq.xlsx",
        "forecast" : "demand_forecast.xlsx",
        "sales"    : "online_retail_CLEANED.xlsx",
    }
    for key, filename in files.items():
        if os.path.exists(filename):
            data[key] = pd.read_excel(filename)
            print(f"  loaded {filename} ({len(data[key]):,} rows)")
        else:
            data[key] = pd.DataFrame()
            print(f"  WARNING: {filename} not found")
    return data

DATA = load_data()
""")

api_lines.append("""
# ════════════════════════════════════════════════════════════
#   PYDANTIC SCHEMAS
#   Define the shape of every request and response.
#   FastAPI uses these to auto-validate all input/output.
# ════════════════════════════════════════════════════════════

class ChurnRequest(BaseModel):
    customer_id: str = Field(..., example="12345")

class ChurnResponse(BaseModel):
    customer_id   : str
    churn_proba   : float
    churn_risk    : str
    segment       : Optional[str]
    recency_days  : Optional[int]
    frequency     : Optional[int]
    monetary      : Optional[float]
    recommendation: str

class DemandRequest(BaseModel):
    stock_code : str = Field(..., example="85123A")
    days_ahead : int = Field(30, ge=1, le=90)

class DemandResponse(BaseModel):
    stock_code    : str
    forecast_days : int
    predictions   : List[dict]
    avg_daily_qty : Optional[float]
    message       : str

class SegmentRequest(BaseModel):
    customer_id: str = Field(..., example="12345")

class SegmentResponse(BaseModel):
    customer_id   : str
    segment       : str
    kmeans_cluster: Optional[int]
    rfm_score     : Optional[float]
    is_vip        : Optional[bool]
    clv_estimate  : Optional[float]
    churn_risk    : Optional[str]

class ReorderRequest(BaseModel):
    stock_code: str = Field(..., example="85123A")

class ReorderResponse(BaseModel):
    stock_code    : str
    eoq           : float
    safety_stock  : float
    reorder_point : float
    avg_daily_qty : float
    abc_class     : Optional[str]
    xyz_class     : Optional[str]
    abc_xyz       : Optional[str]
    is_dead_stock : bool
    stockout_risk : Optional[str]
    recommendation: str
""")

api_lines.append("""
# ════════════════════════════════════════════════════════════
#   HELPER FUNCTIONS
#   Convert numeric scores into plain-language actions.
#   These are what make the API useful for CRM teams.
# ════════════════════════════════════════════════════════════

def get_churn_recommendation(risk: str, segment: str = "") -> str:
    recs = {
        "High"  : "Immediate action: Send win-back offer with 20% discount. Flag for CRM.",
        "Medium": "Proactive outreach: Send loyalty reward email. Suggest related products.",
        "Low"   : "Customer is healthy. Continue engagement via newsletter.",
    }
    return recs.get(str(risk), "Monitor customer activity.")

def get_reorder_recommendation(row: pd.Series) -> str:
    if row.get("IsDeadStock", 0) == 1:
        return "Dead stock: Consider markdown or liquidation. Do not reorder."
    risk = str(row.get("StockoutRisk", ""))
    eoq  = int(row.get("EOQ", 0))
    rop  = int(row.get("ReorderPoint", 0))
    if   risk == "High"  : return f"URGENT: Reorder {eoq} units now. Below safety stock."
    elif risk == "Medium" : return f"Reorder soon: EOQ = {eoq} units. Monitor weekly."
    else                  : return f"Stock healthy. Next reorder at {rop} units remaining."
""")

api_lines.append("""
# ════════════════════════════════════════════════════════════
#   ENDPOINT 1 – Health Check
#   GET /health
#   Always the first endpoint to build. Lets you verify
#   the server is running and data loaded correctly.
# ════════════════════════════════════════════════════════════

@app.get("/health", tags=["System"])
def health_check():
    return {
        "status"   : "healthy",
        "timestamp": datetime.utcnow().isoformat(),
        "data_loaded": {
            "rfm_customers" : len(DATA["rfm"]),
            "inventory_skus": len(DATA["inventory"]),
            "forecast_rows" : len(DATA["forecast"]),
            "sales_rows"    : len(DATA["sales"]),
        }
    }
""")

api_lines.append("""
# ════════════════════════════════════════════════════════════
#   ENDPOINT 2 – Summary KPIs
#   GET /summary
#   Returns headline numbers across all models in one call.
#   Used for the executive dashboard smoke test.
# ════════════════════════════════════════════════════════════

@app.get("/summary", tags=["System"])
def get_summary():
    rfm = DATA["rfm"]
    inv = DATA["inventory"]
    return {
        "generated_at": datetime.utcnow().isoformat(),
        "customers": {
            "total"          : len(rfm),
            "vip"            : int(rfm["IsVIP"].sum())                               if "IsVIP"         in rfm.columns else 0,
            "high_churn_risk": int((rfm["ChurnRisk"].astype(str)=="High").sum())    if "ChurnRisk"     in rfm.columns else 0,
            "avg_churn_proba": round(float(rfm["ChurnProba"].mean()), 4)             if "ChurnProba"    in rfm.columns else 0.0,
        },
        "inventory": {
            "total_skus"        : len(inv),
            "high_stockout_risk": int((inv["StockoutRisk"].astype(str)=="High").sum()) if "StockoutRisk" in inv.columns else 0,
            "dead_stock_skus"   : int(inv["IsDeadStock"].sum())                         if "IsDeadStock"  in inv.columns else 0,
        },
        "forecast": {"rows_available": len(DATA["forecast"])},
    }
""")

api_lines.append("""
# ════════════════════════════════════════════════════════════
#   ENDPOINT 3 – Churn Prediction
#   POST /predict/churn
#   Input : { "customer_id": "12345" }
#   Output: churn_proba, churn_risk, segment, RFM values,
#           plain-language CRM recommendation
# ════════════════════════════════════════════════════════════

@app.post("/predict/churn", response_model=ChurnResponse, tags=["Models"])
def predict_churn(req: ChurnRequest):
    df   = DATA["rfm"]
    if df.empty:
        raise HTTPException(503, "rfm_segments_churn.xlsx not loaded.")
    mask = df["CustomerID"].astype(str) == str(req.customer_id)
    if not mask.any():
        raise HTTPException(404, f"Customer '{req.customer_id}' not found.")
    row        = df[mask].iloc[0]
    churn_risk = str(row.get("ChurnRisk", "Unknown"))
    segment    = str(row.get("Segment",   "")) if "Segment" in df.columns else None
    return ChurnResponse(
        customer_id    = req.customer_id,
        churn_proba    = round(float(row.get("ChurnProba", 0.0)), 4),
        churn_risk     = churn_risk,
        segment        = segment,
        recency_days   = int(row.get("Recency",   0)),
        frequency      = int(row.get("Frequency", 0)),
        monetary       = round(float(row.get("Monetary", 0.0)), 2),
        recommendation = get_churn_recommendation(churn_risk, segment or ""),
    )
""")

api_lines.append("""
# ════════════════════════════════════════════════════════════
#   ENDPOINT 4 – Demand Forecast
#   POST /predict/demand
#   Input : { "stock_code": "85123A", "days_ahead": 30 }
#   Output: list of {date, yhat, yhat_lower, yhat_upper}
#           for the requested forecast horizon
# ════════════════════════════════════════════════════════════

@app.post("/predict/demand", response_model=DemandResponse, tags=["Models"])
def predict_demand(req: DemandRequest):
    df = DATA["forecast"]
    if df.empty:
        raise HTTPException(503, "demand_forecast.xlsx not loaded.")
    mask = df["StockCode"].astype(str) == str(req.stock_code)
    if not mask.any():
        raise HTTPException(404, f"StockCode '{req.stock_code}' not found.")
    sku_fc = df[mask].copy().sort_values("ds").tail(req.days_ahead)
    predictions = [{
        "date"      : str(r["ds"])[:10],
        "yhat"      : round(float(r.get("yhat",       0)), 2),
        "yhat_lower": round(float(r.get("yhat_lower", 0)), 2),
        "yhat_upper": round(float(r.get("yhat_upper", 0)), 2),
    } for _, r in sku_fc.iterrows()]
    avg_qty = None
    inv = DATA["inventory"]
    if not inv.empty:
        im = inv["StockCode"].astype(str) == str(req.stock_code)
        if im.any():
            avg_qty = round(float(inv[im].iloc[0].get("AvgDailyQty", 0)), 2)
    return DemandResponse(
        stock_code    = req.stock_code,
        forecast_days = len(predictions),
        predictions   = predictions,
        avg_daily_qty = avg_qty,
        message       = f"Forecast for {len(predictions)} days returned successfully.",
    )
""")

api_lines.append("""
# ════════════════════════════════════════════════════════════
#   ENDPOINT 5 – Customer Segment Score
#   POST /segment/score
#   Input : { "customer_id": "12345" }
#   Output: segment name, KMeans cluster, RFM score,
#           VIP flag, CLV estimate, churn risk
# ════════════════════════════════════════════════════════════

@app.post("/segment/score", response_model=SegmentResponse, tags=["Models"])
def segment_score(req: SegmentRequest):
    df   = DATA["rfm"]
    if df.empty:
        raise HTTPException(503, "rfm_segments_churn.xlsx not loaded.")
    mask = df["CustomerID"].astype(str) == str(req.customer_id)
    if not mask.any():
        raise HTTPException(404, f"Customer '{req.customer_id}' not found.")
    row = df[mask].iloc[0]
    return SegmentResponse(
        customer_id    = req.customer_id,
        segment        = str(row.get("Segment",       "Unknown")) if "Segment"        in df.columns else "N/A",
        kmeans_cluster = int(row.get("KMeans_Cluster", -1))        if "KMeans_Cluster" in df.columns else None,
        rfm_score      = round(float(row.get("RFM_Score", 0)), 2)  if "RFM_Score"      in df.columns else None,
        is_vip         = bool(row.get("IsVIP", False))              if "IsVIP"          in df.columns else None,
        clv_estimate   = round(float(row.get("CLV_Estimate", 0)), 2) if "CLV_Estimate" in df.columns else None,
        churn_risk     = str(row.get("ChurnRisk", "Unknown"))      if "ChurnRisk"      in df.columns else None,
    )
""")

api_lines.append("""
# ════════════════════════════════════════════════════════════
#   ENDPOINT 6 – Inventory Reorder
#   POST /inventory/reorder
#   Input : { "stock_code": "85123A" }
#   Output: EOQ, safety stock, reorder point, ABC-XYZ class,
#           dead-stock flag, plain-language recommendation
# ════════════════════════════════════════════════════════════

@app.post("/inventory/reorder", response_model=ReorderResponse, tags=["Models"])
def inventory_reorder(req: ReorderRequest):
    df = DATA["inventory"]
    if df.empty:
        raise HTTPException(503, "inventory_eoq.xlsx not loaded.")
    mask = df["StockCode"].astype(str) == str(req.stock_code)
    if not mask.any():
        raise HTTPException(404, f"StockCode '{req.stock_code}' not found.")
    row = df[mask].iloc[0]
    return ReorderResponse(
        stock_code    = req.stock_code,
        eoq           = round(float(row.get("EOQ",          0)), 0),
        safety_stock  = round(float(row.get("SafetyStock",  0)), 0),
        reorder_point = round(float(row.get("ReorderPoint", 0)), 0),
        avg_daily_qty = round(float(row.get("AvgDailyQty",  0)), 2),
        abc_class     = str(row.get("ABC",       "")) if "ABC"        in df.columns else None,
        xyz_class     = str(row.get("XYZ",       "")) if "XYZ"        in df.columns else None,
        abc_xyz       = str(row.get("ABC_XYZ",   "")) if "ABC_XYZ"    in df.columns else None,
        is_dead_stock = bool(row.get("IsDeadStock", 0)),
        stockout_risk = str(row.get("StockoutRisk", "")) if "StockoutRisk" in df.columns else None,
        recommendation= get_reorder_recommendation(row),
    )
""")

api_lines.append("""
# ════════════════════════════════════════════════════════════
#   ENDPOINT 7 – High-Risk Customers Batch
#   GET /predict/churn/high-risk?top_n=20&segment=Champions
#   Returns top-N highest churn probability customers.
#   Used for CRM bulk export and campaign targeting.
# ════════════════════════════════════════════════════════════

@app.get("/predict/churn/high-risk", tags=["Batch"])
def get_high_risk_customers(
    top_n  : int           = Query(20, ge=1, le=200),
    segment: Optional[str] = Query(None, description="Filter by segment name")
):
    df = DATA["rfm"].copy()
    if df.empty:
        raise HTTPException(503, "rfm_segments_churn.xlsx not loaded.")
    if segment and "Segment" in df.columns:
        df = df[df["Segment"].astype(str).str.lower() == segment.lower()]
    top = (df[df["ChurnRisk"].astype(str) == "High"]
             .sort_values("ChurnProba", ascending=False)
             .head(top_n))
    result = [{
        "customer_id" : str(r["CustomerID"]),
        "churn_proba" : round(float(r.get("ChurnProba", 0)), 4),
        "churn_risk"  : str(r.get("ChurnRisk",  "")),
        "segment"     : str(r.get("Segment",    "")) if "Segment" in df.columns else "",
        "monetary"    : round(float(r.get("Monetary", 0)), 2),
        "recency_days": int(r.get("Recency", 0)),
    } for _, r in top.iterrows()]
    return {"total_high_risk": len(result), "customers": result}
""")

api_lines.append("""
# ════════════════════════════════════════════════════════════
#   ENDPOINT 8 – Inventory Reorder Alerts Batch
#   GET /inventory/alerts?abc_class=A&top_n=50
#   Returns SKUs with high stockout risk.
#   Used for automated PO (Purchase Order) generation.
# ════════════════════════════════════════════════════════════

@app.get("/inventory/alerts", tags=["Batch"])
def get_reorder_alerts(
    abc_class: Optional[str] = Query(None, description="Filter: A, B, or C"),
    top_n    : int           = Query(50,   ge=1, le=500)
):
    df = DATA["inventory"].copy()
    if df.empty:
        raise HTTPException(503, "inventory_eoq.xlsx not loaded.")
    if abc_class and "ABC" in df.columns:
        df = df[df["ABC"].astype(str).str.upper() == abc_class.upper()]
    alerts = (df[df["StockoutRisk"].astype(str) == "High"].head(top_n)
              if "StockoutRisk" in df.columns else df.head(top_n))
    result = [{
        "stock_code"   : str(r["StockCode"]),
        "eoq"          : int(r.get("EOQ",           0)),
        "safety_stock" : int(r.get("SafetyStock",   0)),
        "reorder_point": int(r.get("ReorderPoint",  0)),
        "avg_daily_qty": round(float(r.get("AvgDailyQty", 0)), 2),
        "abc_xyz"      : str(r.get("ABC_XYZ", "")) if "ABC_XYZ" in df.columns else "",
        "stockout_risk": str(r.get("StockoutRisk", "")) if "StockoutRisk" in df.columns else "",
    } for _, r in alerts.iterrows()]
    return {"total_alerts": len(result), "skus": result}
""")

# Join all sections and write the file
full_code = '\n'.join(api_lines)
with open('fastapi_app.py', 'w') as f:
    f.write(full_code)

print('✅  fastapi_app.py written successfully!')
print(f'   File size : {len(full_code):,} characters')
print(f'   Endpoints : 8')
print()
print('  ─────────────────────────────────────────')
print('  To start the API, open a NEW terminal and run:')
print()
print('  uvicorn fastapi_app:app --reload --port 8000')
print()
print('  Then open in browser:')
print('  http://localhost:8000/docs   ← Interactive test UI')
print('  http://localhost:8000/health ← Quick health check')
print('  ─────────────────────────────────────────')

✅  fastapi_app.py written successfully!
   File size : 15,575 characters
   Endpoints : 8

  ─────────────────────────────────────────
  To start the API, open a NEW terminal and run:

  uvicorn fastapi_app:app --reload --port 8000

  Then open in browser:
  http://localhost:8000/docs   ← Interactive test UI
  http://localhost:8000/health ← Quick health check
  ─────────────────────────────────────────


## STEP 5 – VERIFY THE FILE WAS WRITTEN CORRECTLY

Before starting the server, confirm `fastapi_app.py` exists and all 8 endpoints are present.

In [5]:
import os

print('\n' + '='*60)
print('  STEP 5 – VERIFYING fastapi_app.py')
print('='*60)

if not os.path.exists('fastapi_app.py'):
    print('  ❌  fastapi_app.py NOT FOUND – re-run Step 4')
else:
    with open('fastapi_app.py', 'r') as f:
        content = f.read()

    file_size = os.path.getsize('fastapi_app.py')
    print(f'  ✅  fastapi_app.py exists  ({file_size:,} bytes)')

    # Check all 8 endpoints are present
    endpoints = [
        ('/health',                    'GET  /health'),
        ('/summary',                   'GET  /summary'),
        ('/predict/churn"',            'POST /predict/churn'),
        ('/predict/demand"',           'POST /predict/demand'),
        ('/segment/score"',            'POST /segment/score'),
        ('/inventory/reorder"',        'POST /inventory/reorder'),
        ('/predict/churn/high-risk"',  'GET  /predict/churn/high-risk'),
        ('/inventory/alerts"',         'GET  /inventory/alerts'),
    ]

    print()
    print('  ── Endpoint verification ──')
    for search_str, label in endpoints:
        found = search_str in content
        icon  = '✅' if found else '❌'
        print(f'  {icon}  {label}')

    print()
    print('  ── Key components ──')
    checks = [
        ('load_data()',          'Data loader function'),
        ('BaseModel',           'Pydantic schemas'),
        ('HTTPException',       'Error handling'),
        ('ChurnResponse',       'Churn response schema'),
        ('ReorderResponse',     'Reorder response schema'),
        ('get_churn_recommendation', 'CRM recommendation helper'),
        ('get_reorder_recommendation', 'Reorder recommendation helper'),
    ]
    for search_str, label in checks:
        found = search_str in content
        icon  = '✅' if found else '❌'
        print(f'  {icon}  {label}')


  STEP 5 – VERIFYING fastapi_app.py
  ✅  fastapi_app.py exists  (18,133 bytes)

  ── Endpoint verification ──
  ✅  GET  /health
  ✅  GET  /summary
  ✅  POST /predict/churn
  ✅  POST /predict/demand
  ✅  POST /segment/score
  ✅  POST /inventory/reorder
  ✅  GET  /predict/churn/high-risk
  ✅  GET  /inventory/alerts

  ── Key components ──
  ✅  Data loader function
  ✅  Pydantic schemas
  ✅  Error handling
  ✅  Churn response schema
  ✅  Reorder response schema
  ✅  CRM recommendation helper
  ✅  Reorder recommendation helper


## STEP 6 – START THE SERVER & TEST ALL ENDPOINTS

### How to start the server

Open a **new terminal** in the same folder and run:

```bash
uvicorn fastapi_app:app --reload --port 8000
```

You should see:
```
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000
```

**Leave that terminal running** and come back here to run the test cells below.

---

> **Note:** If the server is not running, the test cells will show a `ConnectionRefusedError`.  
> That is expected — start the server first, then run the cells.


In [20]:
import requests, json

BASE_URL = "http://localhost:8000"

def pretty(response):
    """Print response status and formatted JSON."""
    print(f'  Status : {response.status_code}')
    try:
        data = response.json()
        print(json.dumps(data, indent=2))
    except:
        print(response.text)

# Load sample IDs from Step 3
rfm         = pd.read_excel('rfm_segments_churn.xlsx')
inv         = pd.read_excel('inventory_eoq.xlsx')
CUSTOMER_ID = str(rfm['CustomerID'].iloc[0])
SKU_CODE    = str(inv['StockCode'].iloc[0])

print(f'  Using CUSTOMER_ID : {CUSTOMER_ID}')
print(f'  Using SKU_CODE    : {SKU_CODE}')
print()
print('  ✅  Test helper ready')
print('  ✅  Make sure the server is running before the next cells')

  Using CUSTOMER_ID : 12346
  Using SKU_CODE    : 85123A

  ✅  Test helper ready
  ✅  Make sure the server is running before the next cells


### Test 1 – Health Check (`GET /health`)

In [22]:
print('\n── GET /health ──────────────────────────────────────')
try:
    r = requests.get(f'{BASE_URL}/health')
    pretty(r)
except requests.exceptions.ConnectionError:
    print('  ❌  Connection refused — is the server running?')
    print('  Run: uvicorn fastapi_app:app --reload --port 8000')


── GET /health ──────────────────────────────────────
  Status : 200
{
  "status": "healthy",
  "timestamp": "2026-06-09T09:39:07.211719",
  "data_loaded": {
    "rfm_customers": 5072,
    "inventory_skus": 4249,
    "forecast_rows": 5619,
    "sales_rows": 652911
  }
}


### Test 2 – Summary KPIs (`GET /summary`)

In [23]:
print('\n── GET /summary ─────────────────────────────────────')
try:
    r = requests.get(f'{BASE_URL}/summary')
    pretty(r)
except requests.exceptions.ConnectionError:
    print('  ❌  Server not running. Start it first.')


── GET /summary ─────────────────────────────────────
  Status : 200
{
  "generated_at": "2026-06-09T09:39:12.442415",
  "customers": {
    "total": 5072,
    "vip": 0,
    "high_churn_risk": 2567,
    "avg_churn_proba": 0.5687
  },
  "inventory": {
    "total_skus": 4249,
    "high_stockout_risk": 2966,
    "dead_stock_skus": 0
  },
  "forecast": {
    "rows_available": 5619
  }
}


### Test 3 – Churn Prediction (`POST /predict/churn`)

In [24]:
print(f'\n── POST /predict/churn  (CustomerID: {CUSTOMER_ID}) ──')
try:
    r = requests.post(
        f'{BASE_URL}/predict/churn',
        json={'customer_id': CUSTOMER_ID}
    )
    pretty(r)
except requests.exceptions.ConnectionError:
    print('  ❌  Server not running.')


── POST /predict/churn  (CustomerID: 12346) ──
  Status : 200
{
  "customer_id": "12346",
  "churn_proba": 0.9618,
  "churn_risk": "High",
  "segment": null,
  "recency_days": 433,
  "frequency": 11,
  "monetary": 372.86,
  "recommendation": "Immediate action: Send win-back offer with 20% discount. Flag for CRM."
}


### Test 4 – Demand Forecast (`POST /predict/demand`)

In [25]:
print(f'\n── POST /predict/demand  (SKU: {SKU_CODE}, 7 days) ──')
try:
    r = requests.post(
        f'{BASE_URL}/predict/demand',
        json={'stock_code': SKU_CODE, 'days_ahead': 7}
    )
    pretty(r)
except requests.exceptions.ConnectionError:
    print('  ❌  Server not running.')


── POST /predict/demand  (SKU: 85123A, 7 days) ──
  Status : 200
{
  "stock_code": "85123A",
  "forecast_days": 7,
  "predictions": [
    {
      "date": "2011-12-28",
      "yhat": -5.46,
      "yhat_lower": -34.22,
      "yhat_upper": 23.67
    },
    {
      "date": "2011-12-29",
      "yhat": 5.96,
      "yhat_lower": -22.0,
      "yhat_upper": 34.94
    },
    {
      "date": "2011-12-30",
      "yhat": -12.64,
      "yhat_lower": -41.22,
      "yhat_upper": 17.65
    },
    {
      "date": "2011-12-31",
      "yhat": -53.85,
      "yhat_lower": -83.75,
      "yhat_upper": -25.66
    },
    {
      "date": "2012-01-01",
      "yhat": -7.15,
      "yhat_lower": -37.38,
      "yhat_upper": 21.12
    },
    {
      "date": "2012-01-02",
      "yhat": -10.51,
      "yhat_lower": -40.36,
      "yhat_upper": 21.2
    },
    {
      "date": "2012-01-03",
      "yhat": -4.02,
      "yhat_lower": -32.29,
      "yhat_upper": 23.99
    }
  ],
  "avg_daily_qty": 46.08,
  "message": "Forecast

### Test 5 – Segment Score (`POST /segment/score`)

In [26]:
print(f'\n── POST /segment/score  (CustomerID: {CUSTOMER_ID}) ──')
try:
    r = requests.post(
        f'{BASE_URL}/segment/score',
        json={'customer_id': CUSTOMER_ID}
    )
    pretty(r)
except requests.exceptions.ConnectionError:
    print('  ❌  Server not running.')


── POST /segment/score  (CustomerID: 12346) ──
  Status : 200
{
  "customer_id": "12346",
  "segment": "N/A",
  "kmeans_cluster": null,
  "rfm_score": 8.0,
  "is_vip": null,
  "clv_estimate": 14.7,
  "churn_risk": "High"
}


### Test 6 – Inventory Reorder (`POST /inventory/reorder`)

In [27]:
print(f'\n── POST /inventory/reorder  (SKU: {SKU_CODE}) ──')
try:
    r = requests.post(
        f'{BASE_URL}/inventory/reorder',
        json={'stock_code': SKU_CODE}
    )
    pretty(r)
except requests.exceptions.ConnectionError:
    print('  ❌  Server not running.')


── POST /inventory/reorder  (SKU: 85123A) ──
  Status : 200
{
  "stock_code": "85123A",
  "eoq": 1510.0,
  "safety_stock": 121.0,
  "reorder_point": 444.0,
  "avg_daily_qty": 46.08,
  "abc_class": "A",
  "xyz_class": "Y",
  "abc_xyz": "AY",
  "is_dead_stock": false,
  "stockout_risk": "Medium",
  "recommendation": "Reorder soon: EOQ = 1510 units. Monitor weekly."
}


### Test 7 – High-Risk Customers Batch (`GET /predict/churn/high-risk`)

In [28]:
print('\n── GET /predict/churn/high-risk?top_n=5 ──')
try:
    r = requests.get(f'{BASE_URL}/predict/churn/high-risk', params={'top_n': 5})
    pretty(r)
except requests.exceptions.ConnectionError:
    print('  ❌  Server not running.')


── GET /predict/churn/high-risk?top_n=5 ──
  Status : 200
{
  "total_high_risk": 5,
  "customers": [
    {
      "customer_id": "17305",
      "churn_proba": 0.9726,
      "churn_risk": "High",
      "segment": "",
      "monetary": 63.5,
      "recency_days": 553
    },
    {
      "customer_id": "18153",
      "churn_proba": 0.9715,
      "churn_risk": "High",
      "segment": "",
      "monetary": 246.22,
      "recency_days": 563
    },
    {
      "customer_id": "18251",
      "churn_proba": 0.9714,
      "churn_risk": "High",
      "segment": "",
      "monetary": 90.0,
      "recency_days": 622
    },
    {
      "customer_id": "13543",
      "churn_proba": 0.9709,
      "churn_risk": "High",
      "segment": "",
      "monetary": 1092.06,
      "recency_days": 538
    },
    {
      "customer_id": "17072",
      "churn_proba": 0.9707,
      "churn_risk": "High",
      "segment": "",
      "monetary": 253.45,
      "recency_days": 529
    }
  ]
}


### Test 8 – Inventory Alerts (`GET /inventory/alerts`)

In [29]:
print('\n── GET /inventory/alerts?abc_class=A&top_n=5 ──')
try:
    r = requests.get(
        f'{BASE_URL}/inventory/alerts',
        params={'abc_class': 'A', 'top_n': 5}
    )
    pretty(r)
except requests.exceptions.ConnectionError:
    print('  ❌  Server not running.')


── GET /inventory/alerts?abc_class=A&top_n=5 ──
  Status : 200
{
  "total_alerts": 5,
  "skus": [
    {
      "stock_code": "47566",
      "eoq": 828,
      "safety_stock": 84,
      "reorder_point": 247,
      "avg_daily_qty": 23.22,
      "abc_xyz": "AY",
      "stockout_risk": "High"
    },
    {
      "stock_code": "21754",
      "eoq": 559,
      "safety_stock": 44,
      "reorder_point": 133,
      "avg_daily_qty": 12.73,
      "abc_xyz": "AY",
      "stockout_risk": "High"
    },
    {
      "stock_code": "22139",
      "eoq": 697,
      "safety_stock": 62,
      "reorder_point": 177,
      "avg_daily_qty": 16.48,
      "abc_xyz": "AY",
      "stockout_risk": "High"
    },
    {
      "stock_code": "22112",
      "eoq": 769,
      "safety_stock": 96,
      "reorder_point": 237,
      "avg_daily_qty": 20.07,
      "abc_xyz": "AZ",
      "stockout_risk": "High"
    },
    {
      "stock_code": "22138",
      "eoq": 641,
      "safety_stock": 57,
      "reorder_point": 154,
      

In [31]:
print('\n' + '='*60)
print('  NEURALRETAIL FASTAPI – NOTEBOOK COMPLETE')
print('='*60)
print()
print('  ✅  Dependencies installed')
print('  ✅  Data files verified')
print('  ✅  Column alignment checked')
print('  ✅  fastapi_app.py written to disk')
print('  ✅  All 8 endpoints verified in file')
print()
print('  Endpoints summary:')
print('  ┌─────────────────────────────────────────────┐')
print('  │ GET  /health                                │')
print('  │ GET  /summary                               │')
print('  │ POST /predict/churn                         │')
print('  │ POST /predict/demand                        │')
print('  │ POST /segment/score                         │')
print('  │ POST /inventory/reorder                     │')
print('  │ GET  /predict/churn/high-risk               │')
print('  │ GET  /inventory/alerts                      │')
print('  └─────────────────────────────────────────────┘')
print()
print('  Start server:  uvicorn fastapi_app:app --reload --port 8000')
print('  API docs:      http://localhost:8000/docs')
print()
print('  Next step → Day 20: Evidently AI Drift Monitoring')


  NEURALRETAIL FASTAPI – NOTEBOOK COMPLETE

  ✅  Dependencies installed
  ✅  Data files verified
  ✅  Column alignment checked
  ✅  fastapi_app.py written to disk
  ✅  All 8 endpoints verified in file

  Endpoints summary:
  ┌─────────────────────────────────────────────┐
  │ GET  /health                                │
  │ GET  /summary                               │
  │ POST /predict/churn                         │
  │ POST /predict/demand                        │
  │ POST /segment/score                         │
  │ POST /inventory/reorder                     │
  │ GET  /predict/churn/high-risk               │
  │ GET  /inventory/alerts                      │
  └─────────────────────────────────────────────┘

  Start server:  uvicorn fastapi_app:app --reload --port 8000
  API docs:      http://localhost:8000/docs

  Next step → Day 20: Evidently AI Drift Monitoring
